# Modern Benchmark Report

Este notebook resume la réplica moderna del TFG y muestra el resultado del benchmark de forma rápida: tabla comparativa, gráficas y ejemplos visuales del conjunto de evaluación.

## Contexto

La réplica moderna compara modelos actuales de `anomalib` sobre `data/mandarins_pynq_cropped` con seeds fijas. La métrica principal es `image_AUROC`, con desempate por `image_AUPR` y después por latencia.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'artifacts').exists() and (candidate / 'README.md').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the project root from the current working directory.')

benchmark_root = PROJECT_ROOT / 'artifacts' / 'modern' / 'benchmark'
final_root = PROJECT_ROOT / 'artifacts' / 'modern' / 'final_model'
leaderboard = pd.read_csv(benchmark_root / 'leaderboard.csv')
runs = pd.read_csv(benchmark_root / 'benchmark_runs.csv')
summary = json.loads((benchmark_root / 'metrics_summary.json').read_text(encoding='utf-8'))
winner_name = summary['winner']['model']
leaderboard

## Ganador del benchmark

Aquí se ve directamente el modelo ganador y sus medias agregadas sobre las tres seeds ejecutadas.

In [ ]:
pd.DataFrame([summary['winner']])

## Gráficas rápidas

Estas dos vistas permiten ver de un vistazo el equilibrio entre calidad predictiva y latencia.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(leaderboard['model'], leaderboard['mean_image_AUROC'], color=['#d97706', '#2563eb'])
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Mean image AUROC')
axes[0].set_ylabel('score')

axes[1].bar(leaderboard['model'], leaderboard['mean_latency_ms'], color=['#b45309', '#1d4ed8'])
axes[1].set_title('Latency per image (ms)')
axes[1].set_ylabel('ms')

fig.tight_layout()
fig

## Resultados por seed

La tabla completa sirve para comprobar la estabilidad del comportamiento del modelo entre particiones.

In [ ]:
runs

## Ejemplos visuales del conjunto final

El pipeline moderno ya genera imágenes exportadas por `anomalib`. Aquí se muestran ejemplos normales y anómalos del split final para que la inspección visual sea inmediata.

In [ ]:
gallery_root = final_root / 'Patchcore' / 'mandarine_cropped_modern' / 'v0' / 'images'
good_examples = sorted((gallery_root / 'good').glob('*'))[:3]
bad_examples = sorted((gallery_root / 'bad').glob('*'))[:3]

print('Good examples:')
for path in good_examples:
    print(path.name)
    display(Image(filename=str(path), width=420))

print('Bad examples:')
for path in bad_examples:
    print(path.name)
    display(Image(filename=str(path), width=420))